In [1]:
import numpy as np
import tensorflow as tf

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [3]:
with open('shakespeare_complete_works.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [4]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<oov>")
tokenizer.fit_on_texts([text])

In [5]:
full_sequence = tokenizer.texts_to_sequences([text])[0]

In [6]:
window_size = 30
input_sequence = []

In [8]:
# using a sliding window
for i in range(len(full_sequence) - window_size):
    input_sequence.append(full_sequence[i : i + window_size + 1])
input_sequences = np.array(input_sequence)

In [9]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1] # maintained as sparse integers

In [10]:
model = Sequential([Embedding(input_dim=vocab_size, output_dim=100, input_length=window_size),
                    LSTM(150),
                    Dense(vocab_size, activation='softmax')])

/opt/anaconda3/envs/env/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-09-18 02:50:12.155287: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-09-18 02:50:12.157537: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-18 02:50:12.157545: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-18 02:50:12.158653: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-18 02:50:12.160322: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [11]:
# compiling with sparse_categorical_crossentropy to save memory on such a big dataset
model.compile(loss = 'sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [15]:
def sample_with_temperature(predictions, temperature = 1.0):
    # higher temp -> more creative (random), lower temp -> more confident
    predictions = np.asarray(predictions).astype('float64')
    predictions = np.log(predictions + 1e-7) / temperature
    exp_preds = np.exp(predictions)
    predictions = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, predictions, 1)
    return np.argmax(probas)



In [14]:
# run the sample_with_temperature after training the model
model.fit(X, y, epochs = 10, batch_size = 256)

Epoch 1/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 507s 75ms/step - accuracy: 0.1333 - loss: 5.2338
Epoch 2/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 303s 45ms/step - accuracy: 0.1431 - loss: 5.0231
Epoch 3/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 299s 44ms/step - accuracy: 0.1515 - loss: 4.8493
Epoch 4/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 292s 43ms/step - accuracy: 0.1604 - loss: 4.6964
Epoch 5/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 208s 31ms/step - accuracy: 0.1699 - loss: 4.5607
Epoch 6/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 214s 32ms/step - accuracy: 0.1797 - loss: 4.4397
Epoch 7/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 202s 30ms/step - accuracy: 0.1900 - loss: 4.3302
Epoch 8/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 203s 30ms/step - accuracy: 0.2001 - loss: 4.2318
Epoch 9/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 203s 30ms/step - accuracy: 0.2097 - loss: 4.1435
Epoch 10/10
6739/6739 ━━━━━━━━━━━━━━━━━━━━ 205s 30ms/step - accuracy: 0.2191 - loss: 4.0641


In [ ]:
def generate_text(seed_text, next_words, temperature = 1.0):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=window_size, padding = 'pre')

        predicted_prob = model.predict(token_list, verbose=0)[0]
        predicted_idx = sample_with_temperature(predicted_prob, temperature)

        output_word = tokenizer.index_word.get(predicted_idx, "")
        seed_text += " " + output_word
    
    return seed_text

In [ ]:
print(generate_text("To be or not to", 10, temperature=0.01))